# BÀI TẬP LỚN: XẤP XỈ HÀM SỐ DÙNG MẠNG MLP (PYTORCH)
**Môn học:** Học sâu (Deep Learning)  
**Nhiệm vụ:** Xây dựng mạng MLP bằng PyTorch để xấp xỉ hàm mật độ xác suất Gaussian 2 chiều.  
**Mục tiêu thực nghiệm:** Khảo sát có kiểm soát 3 nhóm A (quy mô mẫu), B (độ rộng), C (độ sâu) trên CPU.

## 1. Thiết lập Môi trường & Nạp Thư viện

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

if "." not in sys.path:
    sys.path.append(".")

device = torch.device("cpu")
print(f"PyTorch: {torch.__version__} | Thiết bị huấn luyện: {device}")

## 2. Chuẩn bị Dữ liệu & Trực quan hóa (Phụ trách: Thành viên 1)
> *Yêu cầu: Thành viên 1 hoàn thiện `src/data_generator.py` và `src/visualize.py` trước khi chạy cell dưới đây.*

In [ ]:
from src.data_generator import setup_standard_datasets, load_dataset_from_csv, gaussian_2d_pdf
from src.visualize import plot_3d_scatter, plot_ground_truth_surface

# Khởi tạo bộ dữ liệu chuẩn (Test 100 mẫu và Pool 5000 mẫu)
setup_standard_datasets(data_dir="data")

# Vẽ bề mặt Gaussian thật
plot_ground_truth_surface(gaussian_2d_pdf, save_path="figures/ground_truth_surface.png")

# Nạp và trực quan hóa phân tán 3D tập test
X_test, y_test = load_dataset_from_csv("data/test_dataset_100.csv")
plot_3d_scatter(X_test, y_test, title="Tập Test Độc Lập (100 Mẫu)", save_path="figures/test_3d_scatter.png")

## 3. Huấn luyện Thử nghiệm Mô hình Baseline (Phụ trách: Thành viên 1 & 2)
> *Yêu cầu: Hoàn thiện `src/models.py` và `src/trainer.py`.*

In [ ]:
from src.models import GaussianMLP
from src.trainer import create_dataloader, train_mlp_pipeline
from src.visualize import plot_learning_curve

# Nạp dữ liệu train/val
X_pool, y_pool = load_dataset_from_csv("data/train_val_pool.csv")
X_val, y_val = X_pool[:400], y_pool[:400]
X_train, y_train = X_pool[400:1400], y_pool[400:1400] # 1000 mẫu

train_loader = create_dataloader(X_train, y_train, batch_size=32, shuffle=True)
val_loader = create_dataloader(X_val, y_val, batch_size=32, shuffle=False)

# Khởi tạo mô hình Baseline: 1 tầng ẩn, 64 units, ReLU
model = GaussianMLP(input_dim=2, output_dim=1, num_hidden_layers=1, hidden_units=64, activation="relu")
print(model)

# Huấn luyện
res = train_mlp_pipeline(model, train_loader, val_loader, epochs=150, lr=0.005, device=device, verbose=True)
plot_learning_curve(res["history"], title="Đường Cong Huấn Luyện Baseline")

## 4. Chạy Tự Động 3 Nhóm Thực Nghiệm Bắt Buộc (Phụ trách: Thành viên 2 & 3)
> *Sau khi TV2 và TV3 hoàn thiện code trong `src/run_experiment_A.py`, `B`, `C`.*

In [ ]:
# Chạy Thực nghiệm A: Quy mô dữ liệu A (Tập con lồng nhau)
from src.run_experiment_A import run_experiment_A
run_experiment_A()

# Chạy Thực nghiệm B: Độ rộng B
from src.run_experiment_B import run_experiment_B
run_experiment_B()

# Chạy Thực nghiệm C: Độ sâu C (Cố định 32 units/tầng & đo thời gian)
from src.run_experiment_C import run_experiment_C
run_experiment_C()

## 5. Đánh Giá Mô Hình Tối Ưu Toàn Diện Trên Tập Test Độc Lập (Phụ trách: Thành viên 4)
> *Sau khi TV4 hoàn thiện code trong `src/evaluate_best_model.py`.*

In [ ]:
from src.evaluate_best_model import evaluate_best_model
evaluate_best_model()